# Topic Graph (preview): Answering "What Are the Main Themes?"

A **topic graph** groups related chunks into **topics** (also called communities), arranged in a hierarchy. At query time, **topic-aware retrieval samples evidence across topics** instead of returning only the closest vector matches, so broad questions get answers that cover the whole collection.

| | Topic graph | Entity graph ([entity-graph](entity-graph.ipynb)) |
|---|---|---|
| Represents | Themes and their hierarchy | Typed entities and their relationships |
| Good for | "What are the main themes…?", "What topics does this collection cover?" | "Which astronauts launched from sites in Kazakhstan?" |
| Ontology | Not needed | Required |
| Built by | **Client-side**: the Foundry IQ skills plugin | **Server-side**: a graph-enabled Blob knowledge source |

> **There is no server-side topic graph builder in Foundry IQ (as of November 2026).** The service *uses* a prepared topic index at query time (`topic_graph`: none, balanced or diverse), but nothing in a skillset or knowledge source *creates* one. The supported way to build it is the `topic-graph-build` command of the [Foundry IQ skills plugin](../docs/foundry-iq-skills-guide.md), which runs a native builder on your machine and uploads the result.

![Topic graph: chunk vectors are linked into a similarity graph, grouped into named topics and stored as a topic index; queries use topic_graph none, balanced or diverse](../docs/images/topic-graph.svg)

> ### ⚠️ This notebook has a manual step
> **Step 4 is done by you in GitHub Copilot CLI, not by running cells.** Steps 1–3 run as normal; Steps 5 and 6 only produce results after Step 4 is finished. Install the tools in the table below **before** you start, so you don't get stuck halfway.

## Prerequisites

| Requirement | Notes |
|---|---|
| [prepare-dataset.ipynb](prepare-dataset.ipynb) completed | `text_vector` is stored and retrievable with a cosine HNSW profile, as the builder requires |
| Search service with topic graph support in its region | |
| Roles: **Search Service Contributor**, **Search Index Data Contributor**, **Search Index Data Reader** | The plugin requires Microsoft Entra sign-in (`az login`); it doesn't accept API keys |
| GitHub Copilot CLI, Git, Node.js, Python 3.12–3.14, Azure CLI, **Windows x64 or Linux x64/ARM64** | The native topic builder doesn't run on macOS. **On WSL**, Node.js, npm and Azure CLI must be the *Linux* installs, and `libicu` must be installed (see Step 4.1) |
| A chat deployment for naming topics | Chosen during the plugin's planning step |


## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv

In [ ]:
import os
from collections import Counter

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeSourceResultsProcessing,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from azure.search.documents.models import (
    BalancedVectorQueryTopicGraph,
    DiverseVectorQueryTopicGraph,
    NoneVectorQueryTopicGraph,
    VectorizableTextQuery,
)
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
search_credential = AzureKeyCredential(search_api_key) if search_api_key else DefaultAzureCredential()

CONTENT_INDEX = "space-exploration"
VECTOR_FIELD = "text_vector"
KNOWLEDGE_SOURCE = "space-topic-ks"
KNOWLEDGE_BASE = "space-topic-kb"
BROAD_QUESTION = "What are the main themes and eras of space exploration covered in these documents?"

index_client = SearchIndexClient(search_endpoint, search_credential)
content_client = SearchClient(search_endpoint, CONTENT_INDEX, search_credential)

print(f"Search endpoint: {search_endpoint}")

## Step 1: Why a topic graph?

Vector search returns the *k* chunks **closest** to the question. For a broad question the closest chunks tend to sit in the same corner of the embedding space, for example several overview paragraphs about the Space Race, so the evidence is **redundant** and misses other themes (Mars rovers, space telescopes, commercial launchers…).

Let's measure that with a plain vector query: how many of the top 20 chunks come from distinct articles?

In [ ]:
plain = content_client.search(
    vector_queries=[VectorizableTextQuery(text=BROAD_QUESTION, k_nearest_neighbors=20, fields=VECTOR_FIELD)],
    select=["chunk_id", "title"],
    top=20,
)
plain_titles = Counter(doc["title"] for doc in plain)
print(f"Top 20 chunks come from {len(plain_titles)} distinct articles:")
for title, n in plain_titles.most_common():
    print(f"  {n:>2} × {title}")

## Step 2: How a topic graph is built (the concept)

The plugin's native builder follows four steps:

1. **Similarity graph:** each chunk is a node, connected to its nearest neighbours by vector similarity
2. **Community detection:** densely connected groups of chunks become **topics**. Running detection at several resolutions gives a **hierarchy** (broad root topics → narrower sub-topics, at most 100 chunks per leaf)
3. **Centroids:** each topic gets a representative vector, the normalised mean of its chunks' vectors
4. **Naming:** an LLM reads a few chunks closest to each centroid and gives the topic a short title and a one-sentence description

The result is uploaded to a separate **topic index**, stored as a small graph of **subject → predicate → object** rows (one row per fact, not one document per topic):

| Predicate | Meaning | Example |
|---|---|---|
| `HasRootTopic` | The graph's top-level topics | `topicgraph-topic` → `Topic 7` |
| `HasSubTopic` | Parent → child in the hierarchy | `Topic 7` → `Topic 60` (*Falcon 9 Launch Vehicle*) |
| `HasCentroidVector` | The topic's centroid, used at query time | `Topic 7` → a 3,072-dimension vector |
| `prop:title`, `prop:description` | LLM-generated name and summary | *SpaceX and Reusable Launch Systems* |
| `prop:level`, `prop:proportion`, `prop:parentProportion` | Depth in the hierarchy and share of the corpus | `level = 0`, `parentProportion ≈ 0.05` |

The content index's vector field is then linked to the topic index through a **`topic_graph_profile`**. At query time, retrieval gathers a candidate pool (`min_diversity_pool_size`, 250–10,000) and **samples across topics** using the centroids, governed by:

| `topic_graph` | Behaviour |
|---|---|
| `NoneVectorQueryTopicGraph` | Plain vector search |
| `BalancedVectorQueryTopicGraph` | Recommended default; blends similarity with diversity (fixed diversity factor 0.2) |
| `DiverseVectorQueryTopicGraph` | Stronger diversity; tune `diversity_factor` (0–1) |

## Step 3: Create the knowledge source and knowledge base

Create a `searchIndex` knowledge source over the content index, searching the **vector field**. Don't set a stored `topic_graph` default yet: the service rejects it until the vector field is linked to a topic index (Step 4). Later we set `topic_graph` per request. The plugin's retrieval setup step (Step 4) expects this knowledge source and knowledge base to already exist.


In [ ]:
# No topic_graph default here: the service rejects it until text_vector has a topicGraphProfile (Step 4).
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE,
    results_processing=KnowledgeSourceResultsProcessing.NONE,
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=CONTENT_INDEX,
        search_fields=[SearchIndexFieldReference(name=VECTOR_FIELD)],
        source_data_fields=[SearchIndexFieldReference(name=f) for f in ("chunk_id", "title", "chunk")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE,
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge source '{KNOWLEDGE_SOURCE}' and knowledge base '{KNOWLEDGE_BASE}' ready")

## Step 4: Build the topic graph (manual, outside the notebook)

> ### 🛑 ACTION REQUIRED: you do this step in a terminal, not by running cells
> Azure AI Search can **use** a topic graph but can't **build** one. You build it on your own machine with the **Foundry IQ skills plugin** for **GitHub Copilot CLI**, then come back here. **Don't continue to Step 5 until the checklist below is complete.**

### What happens in this step

You'll paste two prompts into Copilot CLI. Each one starts a phase that only **reads** first and then stops at an **approval card** before it changes anything in Azure.

| Phase | You paste | The plugin runs | What changes in Azure |
|---|---|---|---|
| **A. Build the topic index** | Prompt 1 | `doctor` → `inspect` → `plan` → *you approve* → `apply` → `verify` | A **new index** named like `fiq-space-exploration-<id>-topics-<id>` is created. The `space-exploration` content index is **not** touched yet |
| **B. Link it for retrieval** | Prompt 2 | `plan-retrieval` → *you approve* → `configure-retrieval` → `verify-retrieval` | `space-exploration` gets a **topic graph profile** (named like `fiq-space-exploration-<id>-retrieval-<id>`) and `text_vector` points to it. The stored default stays `none`, so nothing changes for existing queries until you ask for `balanced` or `diverse` |

**Inside phase A, `apply` does the heavy lifting on your machine:**

1. **Downloads every chunk vector** from `space-exploration` (about 4,000 chunks × 3,072 dimensions). This is why `text_vector` must be *stored* and *retrievable*.
2. **Builds a nearest-neighbour similarity graph** and runs **community detection** at two resolutions, which gives broad **root topics** and narrower **sub-topics**. Our run produced **170 topics: 39 root topics and 131 sub-topics**.
3. **Computes a centroid vector** for each topic.
4. **Names each topic** by sending a few representative chunks to your chat deployment, which returns a title and a description. This is the only billable model usage, and the approval card caps the number of calls.
5. **Uploads** the topics, hierarchy and centroids to the new topic index (about 1,200 rows), then `verify` reads them back.

**Phase B is separate** because building an index doesn't change retrieval. Only the link from `text_vector` to the topic index does, and it touches your live content index, so it gets its own approval.

### ✅ To-do checklist

| | To do | Where | Done when |
|---|---|---|---|
| ☐ | **4.1** Install the tools (once per machine) | Terminal | All checks in 4.1 pass |
| ☐ | **4.2** Run the code cell below to generate your two prompts | This notebook | Prompts are printed with no `<placeholders>` |
| ☐ | **4.3** Start Copilot CLI and install the plugin | Copilot CLI | `/mcp` shows `azure` and `foundry-iq-graph` connected |
| ☐ | **4.4** Paste **Prompt 1**, approve, wait for the build | Copilot CLI | Copilot reports the new generation is **verified** |
| ☐ | **4.5** Paste **Prompt 2**, approve | Copilot CLI | Copilot reports retrieval is **verified** |
| ☐ | **4.6** Come back and run Step 5 | This notebook | `Linked topic index:` shows an index name |

---

### 4.1 Install the tools (once per machine)

> 👉 **TO DO:** run each check in a terminal and install anything that's missing.

| Requirement | Check with | Notes |
|---|---|---|
| Windows x64 or Linux x64/ARM64 | `uname -m` | The native builder doesn't run on macOS |
| Git | `git --version` | Needed to add the plugin marketplace |
| Node.js + npm | `node --version` | Used by Copilot CLI and the plugin's MCP servers |
| Python 3.12–3.14 with `venv` | `python3 --version` | The plugin creates its own isolated runtime |
| Azure CLI, signed in with Entra ID | `az account show` | The plugin **doesn't accept API keys** |
| GitHub Copilot CLI | `copilot --version` | Install with `npm install -g @github/copilot` |
| Roles on the search service | — | **Search Service Contributor** and **Search Index Data Contributor** (to create and fill the topic index) |

> **Using WSL?** WSL can silently pick up the **Windows** `node`, `npm` and `az` from `/mnt/c/...`, and then Copilot CLI and the plugin don't work. Run `which node npm az`; none of them should start with `/mnt/c/`. If any do, install the Linux versions:
>
> ```bash
> curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.3/install.sh | bash && source ~/.bashrc
> nvm install --lts                                       # Linux Node.js + npm
> curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash  # Linux Azure CLI
> sudo apt install -y libicu-dev                          # needed by the plugin's Azure MCP server
> hash -r && az login --tenant <tenant-id-or-domain>
> ```

### 4.2 Generate your prompts

> 👉 **TO DO:** run the code cell below this one.

The skill never guesses resource names, so you must give them explicitly. The cell reads your `.env`, looks up the resource group and subscription with the Azure CLI, and prints **Prompt 1** and **Prompt 2** ready to paste. If any value still shows as `<placeholder>`, fix it before continuing.

### 4.3 Start Copilot CLI and install the plugin

> 👉 **TO DO:** in a **new terminal** (not inside this notebook), from the repo folder:

```bash
az login --tenant <tenant-id-or-domain>   # skip if already signed in to the right tenant
copilot                                    # starts the interactive Copilot CLI
```

Then, **inside Copilot CLI** (these are slash commands, not shell commands):

```text
/plugin marketplace add microsoft/azure-skills
/plugin install foundry-iq-skills@azure-skills
```

Type `/exit`, start `copilot` again so the new skills load, then type `/mcp` and check that **`azure`** and **`foundry-iq-graph`** are both connected.

### 4.4 Build the topic index (phase A)

> 👉 **TO DO:** paste **Prompt 1** into Copilot CLI, allow the read-only tool calls, then answer **Approve** on the approval card.

What you'll see:

1. **`doctor`**: readiness checks for the runtime, your sign-in, the search service and quotas. Fix anything it reports before continuing.
2. **`inspect`**: confirms `text_vector` is stored, retrievable and uses a cosine HNSW profile (the dataset notebook set it up that way).
3. **`plan`**: the **approval card** shows the headline, estimated time and cost, and the maximum number of model calls. **Approve** it.
4. **`apply`**: the build described above runs. Expect several minutes.
5. **`verify`**: reads every topic row back and reports the new **generation**.

> If the build stops with `MODEL_CALL_BUDGET_EXHAUSTED`, ask Copilot to raise the model call cap and resume; it continues where it stopped.

### 4.5 Turn on topic retrieval (phase B)

> 👉 **TO DO:** once 4.4 reports **verified**, paste **Prompt 2** into Copilot CLI and **Approve** the second card.

The skill runs `plan-retrieval`, shows the card, then `configure-retrieval` and `verify-retrieval`. The knowledge source `space-topic-ks` and knowledge base `space-topic-kb` from Step 3 must already exist, which is why Step 3 comes first.

### 4.6 Come back here

> 👉 **TO DO:** return to this notebook and run Step 5.

If `Linked topic index:` still says `none yet`, step 4.5 hasn't completed. If a cell hangs right after you sign in again with `az login`, interrupt it and run it again.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `copilot` keeps asking *"Install GitHub Copilot CLI?"* (WSL) | `npm` is the Windows one. Install Linux Node.js (see 4.1), then `npm install -g @github/copilot` |
| `Failed to connect to MCP server "azure"` | On Ubuntu/WSL install `libicu-dev`, then `/exit` and restart `copilot` |
| `SOURCE_VECTORS_NOT_RETRIEVABLE` / `NOT_STORED` | Re-run [prepare-dataset.ipynb](prepare-dataset.ipynb), which creates `text_vector` correctly |
| Copilot can't find the search service | Check the subscription in Prompt 1 matches `az account show` |

More detail: [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md).


In [ ]:
import shutil
import subprocess
from urllib.parse import urlparse

search_service = urlparse(search_endpoint).hostname.split(".")[0]
foundry_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "<foundry-endpoint>").split("/api/projects/")[0]
chat_deployment = os.environ.get("GRAPH_LLM_DEPLOYMENT") or os.environ.get("LLM_DEPLOYMENT", "<chat-deployment>")
embedding_model = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-large")

# Resource group and subscription aren't in .env; look them up with the Azure CLI (current subscription only).
resource_group, subscription = "<resource-group>", "<subscription-id>"
if shutil.which("az"):
    found = subprocess.run(
        ["az", "resource", "list", "--name", search_service,
         "--resource-type", "Microsoft.Search/searchServices", "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True,
    ).stdout.strip()
    if found:
        parts = found.split("/")
        subscription, resource_group = parts[2], parts[4]

print(f"Search service:   {search_service}")
print(f"Resource group:   {resource_group}")
print(f"Subscription:     {subscription}")
print(f"Foundry endpoint: {foundry_endpoint}")
print(f"Chat deployment:  {chat_deployment}")
print(f"Embedding model:  {embedding_model}")
if "<" in resource_group:
    print("\n⚠️ Couldn't find the search service in the current subscription. Run `az account set -s <subscription>` "
          "or replace the placeholders below by hand.")

print(f"""
── Prompt 1: plan and build (paste into Copilot CLI) ──
Check whether this machine is ready to build a Topic Graph. Then inspect the source index `{CONTENT_INDEX}` \
on search service `{search_service}` (resource group `{resource_group}`, subscription `{subscription}`) and plan \
a Topic Graph build. Use key field `chunk_id`, text field `chunk`, source field `source_url`, title field `title`, \
vector field `{VECTOR_FIELD}`, embedding model `{embedding_model}`, and name topics with chat deployment \
`{chat_deployment}` on `{foundry_endpoint}`. Show me the approval card before changing anything.

── Prompt 2: enable retrieval (after the build is verified) ──
Turn on topic retrieval for the verified generation using knowledge source `{KNOWLEDGE_SOURCE}` and knowledge \
base `{KNOWLEDGE_BASE}` on search service `{search_service}`. Keep the stored default off.
""")

## Step 5: Inspect what the plugin created

> **Requires Step 4.** Without it, these cells print `none yet` and skip.

The topic index schema is **produced by the plugin** and isn't documented elsewhere, so reading it back is the best way to see what a topic graph actually contains. Three views:

1. **The link:** which topic graph profile and topic index `text_vector` now points to (created in phase B)
2. **The raw index:** its fields and a few sample rows. Each row is one *subject → predicate → object* fact (see the table in Step 2), which is why there are far more rows than topics
3. **The topic tree:** the root topics and their sub-topics, as the LLM named them

### Inspection helpers


In [ ]:
from collections import defaultdict


def linked_topic_index() -> tuple[str | None, str | None]:
    """Return (profile name, topic index name) linked to the content index's vector field."""
    content_index = index_client.get_index(CONTENT_INDEX)
    vector_field = next(f for f in content_index.fields if f.name == VECTOR_FIELD)
    profiles = content_index.vector_search.topic_graph_profiles or []
    profile = next((p for p in profiles if p.name == vector_field.topic_graph_profile), None)
    return vector_field.topic_graph_profile, (profile.index_name if profile else None)


def describe_topic_index(name: str, sample: int = 5) -> None:
    topic_index = index_client.get_index(name)
    print(f"Topic index '{name}' fields:")
    for field in topic_index.fields:
        dims = f"dims={field.vector_search_dimensions}" if field.vector_search_dimensions else ""
        print(f"  {field.name:<26} {str(field.type):<36} {dims}")
    results = SearchClient(search_endpoint, name, search_credential).search(search_text="*", include_total_count=True, top=sample)
    rows = list(results)
    print(f"\n{results.get_count()} rows (subject → predicate → object facts). Sample, empty fields hidden:")
    for doc in rows:
        print({k: v for k, v in doc.items() if not k.startswith("@") and k != "id" and v not in (None, "", [])})


def show_topic_tree(name: str, max_children: int = 4) -> None:
    rows = list(SearchClient(search_endpoint, name, search_credential).search(
        search_text="*",
        filter="predicate eq 'HasRootTopic' or predicate eq 'HasSubTopic'",
        select=["subject_id", "predicate", "object_id", "object_id_display_name"],
        top=1000,
    ))
    roots = [r for r in rows if r["predicate"] == "HasRootTopic"]
    children = defaultdict(list)
    for r in rows:
        if r["predicate"] == "HasSubTopic":
            children[r["subject_id"]].append(r["object_id_display_name"])
    print(f"{len(roots)} root topics, {sum(map(len, children.values()))} sub-topics. Largest root topics first:\n")
    for root in sorted(roots, key=lambda r: -len(children[r["object_id"]])):
        subs = children[root["object_id"]]
        more = f", … (+{len(subs) - max_children})" if len(subs) > max_children else ""
        print(f"• {root['object_id_display_name']}")
        if subs:
            print(f"    └ {', '.join(subs[:max_children])}{more}")


### Run it

In [ ]:
PROFILE_NAME, TOPIC_INDEX = linked_topic_index()
print(f"{VECTOR_FIELD}.topic_graph_profile = {PROFILE_NAME}")
print(f"Linked topic index: {TOPIC_INDEX or 'none yet — complete Step 4 first'}")

In [ ]:
if TOPIC_INDEX:
    describe_topic_index(TOPIC_INDEX)

In [ ]:
if TOPIC_INDEX:
    show_topic_tree(TOPIC_INDEX)


## Step 6: Compare `none`, `balanced` and `diverse`

> **Requires Step 4.** Without it, the comparison is skipped.

Send the same broad question three times through the knowledge base, overriding `topic_graph` per request, and measure how many **distinct articles** the evidence covers.

### Comparison helpers


In [ ]:
kb_client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=KNOWLEDGE_BASE)

TOPIC_SETTINGS = {
    "none": NoneVectorQueryTopicGraph(),
    "balanced": BalancedVectorQueryTopicGraph(min_diversity_pool_size=1000),
    "diverse (0.5)": DiverseVectorQueryTopicGraph(diversity_factor=0.5, min_diversity_pool_size=1000),
}


def retrieve_with_topics(topic_graph) -> list[dict]:
    request = KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=BROAD_QUESTION)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
            topic_graph=topic_graph,
        )],
    )
    return [dict(ref.source_data or {}) for ref in kb_client.retrieve(request).references or []]


def compare_topic_settings() -> None:
    for label, setting in TOPIC_SETTINGS.items():
        refs = retrieve_with_topics(setting)
        titles = Counter(r.get("title") for r in refs)
        print(f"{label:<14} {len(refs):>2} chunks from {len(titles):>2} articles: {[t for t, _ in titles.most_common(6)]}")

### Run it

In [ ]:
if TOPIC_INDEX:
    compare_topic_settings()
else:
    print("No topic index linked yet; complete Step 4 first.")

With `balanced` and `diverse`, evidence spreads across more articles and themes instead of concentrating on a few overview articles. In our run:

| Setting | Distinct articles in ~25 chunks | What changed |
|---|---|---|
| `none` | 9 | Dominated by *Space exploration*, *Space Race* and *NASA* overviews |
| `balanced` | 12 | Adds specific themes such as *Exploration of Mars* and *Goddard Space Flight Center* |
| `diverse (0.5)` | 16 | Reaches *Hubble Space Telescope*, *Lunar Gateway* and more |

Your exact titles will vary between runs and topic graph generations, but the trend should hold.

## Clean up

Delete the knowledge base and knowledge source created here. Retire the topic index through the plugin (*"retire my topic graph staging"*) so its own state stays consistent; deleting it by hand would leave the content index's `topic_graph_profile` pointing at a missing index.


In [ ]:
# index_client.delete_knowledge_base(KNOWLEDGE_BASE)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge base and knowledge source")

## Summary

- **Why:** plain vector search returns near-duplicate evidence for broad questions; topic sampling spreads evidence across themes
- **How:** similarity graph → communities (hierarchy) → centroids → LLM titles and descriptions → topic index, stored as subject → predicate → object rows
- **Who builds it:** you, on your machine, with the Foundry IQ skills plugin in GitHub Copilot CLI. Two approved phases: **build** the topic index, then **link** it to `text_vector` through a `topic_graph_profile`
- **Query time:** `topic_graph` on the knowledge source (stored default) or per request: none, balanced (recommended) or diverse

### Next steps

- [entity-graph.ipynb](entity-graph.ipynb): relationship questions with an entity graph
- [docs/foundry-iq-skills-guide.md](../docs/foundry-iq-skills-guide.md): the full plugin workflow
